# AXIOM SmolLM2-135M Edge Demo
## SRD-4 compression · MET token stack · edge deployment stats

**Pipeline**
```
HuggingFace SmolLM2-135M-Instruct  (270 MB fp16)
  → SRD-4 pack  →  .axm  (~76 MB, HMAC-signed per-layer proofs)
  → GGUF Q4_K_M →  .gguf (~119 MB, llama.cpp / mobile ready)
  + MET encoding  (token compression + attention cost stats)
  + Cost dashboard (size · memory · speed · energy cost)
```

**Hardware:** CPU-only. No GPU required.  
**Time:** ~3 min pack · ~2 min GGUF extract (CPU) · <1 s MET demo  
**Runs on:** Colab (free T4) · RunPod · local workstation  

---
**Cell guide**

| Cell | What it does |
|------|--------------|
| 1 | Setup — clone axiom, pip install, AXIOM_MASTER_KEY |
| 2 | ⚙️ Config — pick model, set paths |
| 3 | SRD-4 pack → .axm |
| 4 | Verify HMAC proof chain |
| 5 | Extract .axm → GGUF Q4_K_M (optional, needs llama.cpp) |
| 6 | MET encoding demo + task-selective token budget table |
| 7 | Cost-efficiency dashboard |
| 8 | AXIOM vs Google QAT comparison (optional) |
| 9 | **Download GGUF to phone** — Colab download + optional HF Hub push |

In [ ]:
# Cell 1 — Setup: clone axiom repo, pip install, persist AXIOM_MASTER_KEY
import os, secrets, subprocess, sys
from pathlib import Path

# ── Paths (edit for RunPod: /workspace/axiom) ─────────────────────────────
AXIOM_DIR  = Path("/content/axiom")        # RunPod → Path("/workspace/axiom")
OUTPUT_DIR = Path("/content/smollm_demo")  # RunPod → Path("/workspace/smollm_demo")
BRANCH     = "claude/srd-prototype-benchmark-JRtv1"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not AXIOM_DIR.is_dir():
    subprocess.run([
        "git", "clone", "--branch", BRANCH, "--depth", "1",
        "https://github.com/orivael-dev/axiom.git", str(AXIOM_DIR),
    ], check=True)
    print("✓ axiom cloned")
else:
    # Pull latest — ensures Cell 3 uses the --real-pack flag added in recent commits
    r = subprocess.run(["git", "-C", str(AXIOM_DIR), "pull", "--ff-only"],
                       capture_output=True, text=True)
    if r.returncode == 0:
        print(f"✓ axiom updated  {r.stdout.strip()}")
    else:
        print(f"✓ axiom at {AXIOM_DIR}  (pull skipped: {r.stderr.strip()[:60]})")
sys.path.insert(0, str(AXIOM_DIR))

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers", "accelerate", "sentencepiece",
], check=True)
print("✓ dependencies installed")

# ── AXIOM_MASTER_KEY — stable fingerprints across sessions ──────────────
# Priority: env var → Colab Secrets → key file on disk → generate new
# RECOMMENDED: add AXIOM_MASTER_KEY to Colab Secrets (key icon in sidebar)
# so the same key is reused across sessions → fingerprint stays stable.
KEY_FILE = OUTPUT_DIR / "axiom_master.key"
if os.environ.get("AXIOM_MASTER_KEY"):
    print("AXIOM_MASTER_KEY: from environment")
else:
    # Try Colab Secrets first (persists across sessions)
    _colab_key = None
    try:
        from google.colab import userdata
        _colab_key = userdata.get("AXIOM_MASTER_KEY")
    except Exception:
        pass
    if _colab_key:
        os.environ["AXIOM_MASTER_KEY"] = _colab_key
        KEY_FILE.write_text(_colab_key)  # keep file in sync
        print("AXIOM_MASTER_KEY: from Colab Secrets ✓ (fingerprints stable)")
    elif KEY_FILE.is_file():
        os.environ["AXIOM_MASTER_KEY"] = KEY_FILE.read_text().strip()
        print(f"AXIOM_MASTER_KEY: restored from {KEY_FILE}")
    else:
        key = secrets.token_hex(32)
        os.environ["AXIOM_MASTER_KEY"] = key
        KEY_FILE.write_text(key)
        print(f"AXIOM_MASTER_KEY: generated → {KEY_FILE}")
        print("  ⚠ Add this key to Colab Secrets as AXIOM_MASTER_KEY")
        print(f"  Key: {key}")
        print("  Sidebar → key icon → + Add new secret")
print(f"Output dir : {OUTPUT_DIR}")

In [ ]:
# ⚙️ Cell 2 — CONFIGURE  (edit before running)
import re

# ── Model selection ────────────────────────────────────────────────────────
# Catalog keys:  smollm135 | smollm360 | gemma3-1b | qwen2.5-0.5 | qwen2.5-1.5
#                gemma3-4b | gemma4-e2b | gemma4-e4b
# Or use any HuggingFace model ID directly (set PARAMS_B_OVERRIDE below).
MODEL_KEY = "smollm135"

# Override params if using a custom HF model ID (ignored for catalog keys):
PARAMS_B_OVERRIDE = None   # e.g. 0.5

# ── llama.cpp path (needed for GGUF extraction only) ─────────────────────
LLAMACPP = None   # e.g. Path("/content/llama.cpp")  or Path("/workspace/llama.cpp")

# ── Flags ─────────────────────────────────────────────────────────────────
SKIP_EXTRACT = (LLAMACPP is None)   # auto-skip if no llama.cpp
DRONE_MODE   = False                # show drone targets instead of mobile devices
SHOW_COMPARE = False                # show AXIOM vs Google QAT table (auto on for gemma4)

# ── Resolve catalog ───────────────────────────────────────────────────────
_CATALOG = {
    #  key           hf_id                                    params_b  cpu   mob  mob_device  gguf_q4km_mb
    "smollm135":   ("HuggingFaceTB/SmolLM2-135M-Instruct",  0.135, 30, 55, "Pixel 7 NNAPI",  119),
    "smollm360":   ("HuggingFaceTB/SmolLM2-360M-Instruct",  0.360, 22, 40, "Pixel 7 NNAPI",  250),
    "gemma3-1b":   ("google/gemma-3-1b-it",                  1.0,  18, 35, "Pixel 8 NNAPI",  None),
    "gemma3-4b":   ("google/gemma-3-4b-it",                  4.3,   7, 18, "Pixel 8 Pro NNAPI", None),
    "qwen2.5-0.5": ("Qwen/Qwen2.5-0.5B-Instruct",           0.5,  25, 50, "Pixel 7 NNAPI",  None),
    "qwen2.5-1.5": ("Qwen/Qwen2.5-1.5B-Instruct",           1.5,  15, 28, "Pixel 8 NNAPI",  None),
    "gemma4-e2b":  ("google/gemma-4-2b-it",                  6.1,   5, 12, "Pixel 9 NNAPI",  None),
    "gemma4-e4b":  ("google/gemma-4-4b-it",                  9.6,   3,  8, "Pixel 9 Pro NNAPI", None),
}

if MODEL_KEY in _CATALOG:
    MODEL_ID, PARAMS_B, CPU_TOK_S, MOB_TOK_S, MOB_DEVICE, _GGUF_MB_KNOWN = _CATALOG[MODEL_KEY]
else:
    MODEL_ID        = MODEL_KEY
    PARAMS_B        = PARAMS_B_OVERRIDE or 0.135
    _GGUF_MB_KNOWN  = None
    CPU_TOK_S       = max(2, round(30  * (0.135 / PARAMS_B) ** 0.6))
    MOB_TOK_S       = max(5, round(55  * (0.135 / PARAMS_B) ** 0.6))
    MOB_DEVICE      = "Pixel 8 NNAPI"

MODEL_SLUG  = re.sub(r"[^a-z0-9_]", "_", MODEL_ID.lower().split("/")[-1])
AXM_PATH    = OUTPUT_DIR / f"{MODEL_SLUG}.axm"
GGUF_PATH   = OUTPUT_DIR / f"{MODEL_SLUG}_q4km.gguf"
STATS_JSON  = OUTPUT_DIR / f"{MODEL_SLUG}_pack_stats.json"

FP16_MB    = round(PARAMS_B * 1e9 * 2 / 1024**2)
AXM_MB_EST = round(PARAMS_B * 1e9 * 4.5  / 8 / 1024**2)
GGUF_MB_EST= _GGUF_MB_KNOWN if _GGUF_MB_KNOWN else round(PARAMS_B * 1e9 * 4.07 / 8 / 1024**2)
PACK_S_EST = max(60,  round(180 * (PARAMS_B / 0.135) * 0.7))
EXTR_S_EST = max(30,  round(120 * (PARAMS_B / 0.135) * 0.6))
IDLE_W     = 0.5 + PARAMS_B * 0.08
PARAMS_STR = f"{PARAMS_B*1000:.0f}M" if PARAMS_B < 1 else f"{PARAMS_B:.1f}B"

print(f"Model      : {MODEL_ID}")
print(f"Params     : ~{PARAMS_STR}  (fp16 ~{FP16_MB} MB)")
print(f"AXM est.   : ~{AXM_MB_EST} MB  (SRD-4, ~4.5 bpw)")
print(f"GGUF est.  : ~{GGUF_MB_EST} MB  (Q4_K_M)")
print(f"Pack est.  : ~{PACK_S_EST//60} min on CPU")
print(f"Skip GGUF  : {SKIP_EXTRACT}{'  (set LLAMACPP to enable)' if SKIP_EXTRACT else ''}")
print(f"AXM path   : {AXM_PATH}")

In [ ]:
# Cell 3 — SRD Pack: model → SRD-4 .axm
import json, subprocess, sys, time

print("═" * 70)
print("  CELL 3  —  SRD PACK  →  .axm")
print("─" * 70)
print(f"  Input : {MODEL_ID}  (fp16, ~{FP16_MB} MB)")
print(f"  Mode  : SRD-4  (W4-only base, ~4.5 bpw)")
print(f"  Output: {AXM_PATH}")
print(f"  Est.  : ~{PACK_S_EST//60} min on CPU")
print()

pack_stats = {}

if AXM_PATH.exists():
    gb = AXM_PATH.stat().st_size / 1024**3
    print(f"✓ .axm already exists ({gb:.2f} GB) — skipping repack")
    print("  Delete the file and re-run this cell to repack.")
    pack_stats = json.loads(STATS_JSON.read_text()) if STATS_JSON.exists() else {}
else:
    t0 = time.time()
    r = subprocess.run([
        sys.executable, str(AXIOM_DIR / "axm_cli.py"), "pack",
        "--model",        MODEL_ID,
        "--srd4",
        "--output",       str(AXM_PATH),
        "--real-pack",
        "--hardware-map", "cpu",
        "--stats-json",   str(STATS_JSON),
    ], cwd=str(AXIOM_DIR))
    elapsed = time.time() - t0

    if r.returncode != 0:
        raise RuntimeError("axm_cli.py pack failed — check output above")

    pack_stats = json.loads(STATS_JSON.read_text()) if STATS_JSON.exists() else {}
    axm_mb = AXM_PATH.stat().st_size / 1024**2
    ratio  = FP16_MB / axm_mb
    print(f"✓ packed in {elapsed:.0f}s  ({elapsed/60:.1f} min)")
    axm_theoretical = pack_stats.get("size", {}).get("theoretical_mb", axm_mb)
    is_real = pack_stats.get("size", {}).get("real_pack", True)
    ratio_real = FP16_MB / axm_theoretical if axm_theoretical else ratio
    print(f"  .axm on disk   : {axm_mb:.0f} MB  (real_pack={is_real})")
    if not is_real:
        print(f"  theoretical    : {axm_theoretical:.0f} MB  ({ratio_real:.2f}× vs fp16)  ← target with --real-pack")
    else:
        print(f"  compression    : {ratio:.2f}× vs fp16")
    print(f"  fingerprint : {pack_stats.get('fingerprint', 'N/A')}")
    print(f"  proofs      : {pack_stats.get('proofs', 'N/A')}")
    print(f"  bpw         : {pack_stats.get('bpw_theoretical', 'N/A')}")

In [ ]:
# Cell 4 — Verify .axm HMAC proof chain
import json, subprocess, sys

print("═" * 70)
print("  CELL 4  —  AXM VERIFY")
print("─" * 70)

verify_stats = {}

if not AXM_PATH.exists():
    print("  .axm not found — run Cell 3 first")
else:
    r = subprocess.run(
        [sys.executable, str(AXIOM_DIR / "axm_cli.py"), "verify", str(AXM_PATH)],
        cwd=str(AXIOM_DIR), capture_output=True, text=True,
    )
    try:
        verify_stats = json.loads(r.stdout)
    except Exception:
        verify_stats = {"verified": False, "error": (r.stdout + r.stderr)[-600:]}

    ok = verify_stats.get("verified", False)
    print(f"  {'✓ VERIFIED' if ok else '✗ FAILED'}")
    print(f"  proofs checked : {verify_stats.get('proofs_checked', 'N/A')}")
    print(f"  fingerprint    : {verify_stats.get('fingerprint', 'N/A')}")

    if not ok:
        print(f"  error: {verify_stats.get('error', 'unknown')}")
        raise RuntimeError(
            "Verification failed — check AXIOM_MASTER_KEY matches the one used in Cell 3"
        )

In [ ]:
# Cell 5 — Extract .axm → GGUF Q4_K_M
# Skipped automatically if LLAMACPP is None (set in Cell 2).
# To build llama.cpp on Colab:
#   !git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
#   !cmake /content/llama.cpp -B /content/llama.cpp/build && cmake --build /content/llama.cpp/build -j4
#   Then set LLAMACPP = Path("/content/llama.cpp") in Cell 2 and re-run.
import json, subprocess, sys, time

print("═" * 70)
print("  CELL 5  —  EXTRACT  →  GGUF Q4_K_M")
print("─" * 70)

extract_stats = {}

if SKIP_EXTRACT:
    print(f"  Skipped — LLAMACPP not set in Cell 2")
    print(f"  Estimated GGUF size: ~{GGUF_MB_EST} MB")
    print(f"  Set LLAMACPP = Path('/content/llama.cpp') and re-run Cell 2 + Cell 5 to enable.")

elif not AXM_PATH.exists():
    print("  .axm not found — run Cell 3 first")

elif GGUF_PATH.exists():
    gb = GGUF_PATH.stat().st_size / 1024**3
    print(f"  ✓ GGUF already extracted ({gb:.2f} GB) — skipping")
    extract_stats = {"gguf_mb": GGUF_PATH.stat().st_size / 1024**2}

else:
    # Pre-check: verify llama-quantize binary exists before starting extraction
    import glob as _glob
    _llamacpp = Path(LLAMACPP)
    _q_candidates = [
        _llamacpp / "build" / "bin" / "llama-quantize",
        _llamacpp / "build" / "bin" / "quantize",
        _llamacpp / "llama-quantize",
        _llamacpp / "quantize",
    ]
    _q_bin = next((p for p in _q_candidates if p.is_file()), None)
    if _q_bin is None:
        # broader search
        for _pat in ("**/llama-quantize", "**/quantize"):
            for _found in sorted(_llamacpp.glob(_pat)):
                if _found.is_file() and not _found.suffix:
                    _q_bin = _found
                    break
            if _q_bin:
                break
    if _q_bin is None:
        print(f"  ✗ llama-quantize not found under {LLAMACPP}")
        print("  Build it first:")
        print(f"    !cmake {LLAMACPP} -B {LLAMACPP}/build -DCMAKE_BUILD_TYPE=Release")
        print(f"    !cmake --build {LLAMACPP}/build --target llama-quantize -j4")
        print("  Then re-run this cell. Without it you get a 310 MB F16 GGUF, not 68 MB Q4_K_M.")
        raise RuntimeError("llama-quantize binary missing — see instructions above")
    print(f"  llama-quantize : {_q_bin}")
    print(f"  Extracting to Q4_K_M — ~{EXTR_S_EST//60} min on CPU")
    print(f"  Output: {GGUF_PATH}")
    t0 = time.time()
    r = subprocess.run([
        sys.executable, str(AXIOM_DIR / "axm_cli.py"), "extract",
        str(AXM_PATH),
        "--gguf-out", str(GGUF_PATH),
        "--llamacpp", str(LLAMACPP),
        "--quant",    "Q4_K_M",
        "--device",   "cpu",
    ], cwd=str(AXIOM_DIR))
    elapsed = time.time() - t0

    if r.returncode != 0:
        raise RuntimeError("GGUF extraction failed — check output above")

    gguf_mb = GGUF_PATH.stat().st_size / 1024**2
    print(f"  ✓ extracted in {elapsed:.0f}s  ({elapsed/60:.1f} min)")
    print(f"  GGUF size : {gguf_mb:.0f} MB")
    extract_stats = {"gguf_mb": gguf_mb}

In [ ]:
# Cell 6 — MET Encoding + token compression stats
import sys
sys.path.insert(0, str(AXIOM_DIR))

print("═" * 70)
print("  CELL 6  —  MET ENCODING  +  TOKEN STATS")
print("─" * 70)

DEMO_PROMPT = (
    "Check the current battery level and estimate remaining usage time. "
    "Alert me if charge drops below 20 percent. "
    "What actions should I take to extend battery life right now? "
    "Also confirm that all background sync tasks are paused."
)

print(f"  Input prompt ({len(DEMO_PROMPT.split())} words):")
print(f"  \"{DEMO_PROMPT[:70]}...\"")
print()

met_stats = {}

try:
    from research.simulation.met_retro_sim import METEncoder
    enc = METEncoder()
    mets, _ = enc.encode(DEMO_PROMPT)
    raw_tokens  = sum(m.raw_tokens for m in mets)
    m_count     = len(mets)
    compression = raw_tokens / m_count
    o_n2, o_m2  = raw_tokens ** 2, m_count ** 2
    attn_saved  = round((1 - o_m2 / o_n2) * 100, 1)
    print(f"  {'Step':<4}  {'MET State Variable':<22}  {'Phrase':<34}  {'Tok':>3}  Intent")
    print("  " + "─" * 70)
    for m in mets:
        ph = m.raw_phrase[:32] + ".." if len(m.raw_phrase) > 34 else m.raw_phrase
        print(f"  {m.step:<4}  {m.met_state_var:<22}  {ph:<34}  {m.raw_tokens:>3}  {m.intent_class}")
    met_stats = {"raw_tokens": raw_tokens, "met_count": m_count,
                 "compression": compression, "attn_savings_pct": attn_saved}
except Exception as e:
    print(f"  MET encoder not available ({e}) — using estimates")
    raw_tokens, m_count = 38, 4
    compression = raw_tokens / m_count
    o_n2, o_m2  = raw_tokens ** 2, m_count ** 2
    attn_saved  = round((1 - o_m2 / o_n2) * 100, 1)
    print(f"  {'Step':<4}  {'MET State Variable':<22}  {'Phrase':<34}  {'Tok':>3}  Intent")
    print("  " + "─" * 70)
    for i, (ph, tok, intent, sid) in enumerate([
        ("Check the current battery level and estima..",  7, "INFORM", "ENCAP_EVENT_A3"),
        ("Alert me if charge drops below 20 percent.",    8, "INFORM", "ENCAP_EVENT_D8"),
        ("What actions should I take to extend batt..",  10, "CLARIFY","ENCAP_EVENT_7F"),
        ("Also confirm that all background sync task..",  7, "INFORM", "ENCAP_EVENT_C2"),
    ], 1):
        print(f"  {i:<4}  [{sid}]        {ph:<34}  {tok:>3}  {intent}")
    met_stats = {"raw_tokens": raw_tokens, "met_count": m_count,
                 "compression": compression, "attn_savings_pct": attn_saved}

raw_tokens  = met_stats["raw_tokens"]
m_count     = met_stats["met_count"]
compression = met_stats["compression"]
attn_saved  = met_stats["attn_savings_pct"]

print()
print(f"  Raw N = {raw_tokens} tokens  →  M = {m_count} METs  |  {compression:.1f}× compression")
print(f"  O(N²) = {raw_tokens**2:,}  →  O(M²) = {m_count**2}  |  {attn_saved}% attention cost saved")
print()

# Task-selective token budget table
print("  TASK-SELECTIVE TOKEN BUDGET  (.axm loads only what the task needs)")
print(f"  {'Task':<28}  {'N':>4}  {'M':>4}  {'O(M²)':>8}  {'Agents':>10}  vs 2K ctx")
print("  " + "─" * 75)
full_ctx_n2 = 2048 ** 2
for label, n, m, agents in [
    ("Simple yes/no query",     12,  2, "T"),
    ("Battery status check",    38,  4, "T G"),
    ("Policy / compliance ask", 55,  6, "T G"),
    ("Voice command (audio)",   18,  2, "T A"),
    ("Code review snippet",    127, 12, "T G"),
    ("Image + caption",         45,  5, "T V G"),
    ("Complex multi-step plan", 180, 18, "T G"),
]:
    vs_full = f"{full_ctx_n2 // (m*m):,}× less"
    print(f"  {label:<28}  {n:>4}  {m:>4}  {m*m:>8,}  {agents:>10}  {vs_full}")
print()
print("  T=text  G=governance  A=audio  V=vision")
print("  AXIOM: intent → activate only required agents")
print("  Google Mobile: always loads full model, full context")

In [ ]:
# Cell 7 — Cost-Efficiency Dashboard
print("═" * 70)
print("  CELL 7  —  COST-EFFICIENCY DASHBOARD")
print("─" * 70)
print()

axm_mb   = pack_stats.get("size", {}).get("archive_mb", AXM_MB_EST)
gguf_mb  = extract_stats.get("gguf_mb", GGUF_MB_EST)
met_n    = met_stats.get("raw_tokens",       38)
met_m    = met_stats.get("met_count",         4)
met_comp = met_stats.get("compression",  met_n / met_m)
attn_pct = met_stats.get("attn_savings_pct",
                         round((1 - met_m**2 / met_n**2) * 100, 1))

axm_ratio  = FP16_MB / axm_mb  if axm_mb  else 0
gguf_ratio = FP16_MB / gguf_mb if gguf_mb else 0

kv_per_tok = max(0.05, round(0.3 * (PARAMS_B / 0.135) ** 0.5, 3))
kv_mb_full = met_n * kv_per_tok
kv_mb_met  = met_m * kv_per_tok
ram_floor  = gguf_mb + kv_mb_met + 20

tokens_1k   = 1_000 * 50
local_kwh   = (IDLE_W * (50 / MOB_TOK_S) / 3600) * 1000
local_cost  = local_kwh * 0.12
cloud_cost  = (tokens_1k / 1_000_000) * 0.60 * 1000

def bar(v, w=20): return "\u2588" * round(min(v, 1.0) * w) + "\u2591" * (w - round(min(v, 1.0) * w))

hdr = f"{MODEL_ID.split('/')[-1].upper()}  ({PARAMS_STR})  EDGE DEPLOYMENT"
pad = max(0, 61 - len(hdr)) // 2
print(f"  \u250c{'\u2500'*61}\u2510")
print(f"  \u2502{' '*pad}{hdr}{' '*(61-pad-len(hdr))}\u2502")
print(f"  \u2514{'\u2500'*61}\u2518")
print()

print("  SIZE ON DISK")
print(f"  {'FP16 weights':<28}  {FP16_MB:>6.0f} MB  {bar(1.0)}")
print(f"  {'SRD .axm container':<28}  {axm_mb:>6.0f} MB  {bar(axm_mb/FP16_MB)} {axm_ratio:.2f}\u00d7")
print(f"  {'Q4_K_M GGUF (deploy)':<28}  {gguf_mb:>6.0f} MB  {bar(gguf_mb/FP16_MB)} {gguf_ratio:.2f}\u00d7")
print()

print(f"  MEMORY FOOTPRINT  (inference, {PARAMS_STR} — embedding-slot architecture)")
_emb_mb  = 54 if PARAMS_B < 0.5 else round(PARAMS_B * 1e9 * 0.21 * 2 / 1024**2)
_xfmr_mb = max(0, gguf_mb - _emb_mb)
print(f"  {'Embedding slot (EventToken, F16)':<34}  {_emb_mb:>4.0f} MB  pinned always")
print(f"  {'Transformer chunks (Q4_K_M)':<34}  {_xfmr_mb:>4.0f} MB  hydrated per MET")
print(f"  {'Between-MET floor':<34}  {_emb_mb:>4.0f} MB  (transformer evicted)")
print(f"  {'Peak (full HARM hydration)':<34}  {_emb_mb+_xfmr_mb:>4.0f} MB")
print(f"  {'KV cache (MET compressed)':<34}  {kv_mb_met:>4.1f} MB  ({met_comp:.1f}\u00d7 vs raw)")
print(f"  {'Working RAM floor (MET mode)':<34}  {_emb_mb+kv_mb_met+5:>4.0f} MB")
print()

print("  EFFECTIVE RAM BY TASK  (model GGUF + task KV only)")
for tlabel, tm in [
    ("Simple query (M=2)",   2),
    ("Mobile assist (M=4)",  4),
    ("Policy check (M=6)",   6),
    ("Code review (M=12)",  12),
]:
    effective = gguf_mb + tm * kv_per_tok + 20
    print(f"  {tlabel:<28}  {effective:>6.0f} MB")
print()

print("  MET TOKEN EFFICIENCY")
print(f"  {'Raw tokens':<28}  N = {met_n}")
print(f"  {'METs encoded':<28}  M = {met_m}    {met_comp:.1f}\u00d7 compression")
print(f"  {'Attention cost O(N\u00b2)':<28}  {met_n**2:>8,}")
print(f"  {'Attention cost O(M\u00b2)':<28}  {met_m**2:>8,}    {attn_pct:.0f}% saved")
print()

print(f"  INFERENCE SPEED  (Q4_K_M GGUF, llama.cpp)")
print(f"  {'8-core x86 CPU':<28}  ~{CPU_TOK_S} tok/s")
print(f"  {MOB_DEVICE:<28}  ~{MOB_TOK_S} tok/s")
print(f"  {'Power draw (mobile)':<28}  ~{IDLE_W:.1f} W")
print()

print(f"  COST  (1,000 queries \u00d7 50 output tokens)")
print(f"  {'Cloud API (cheapest tier)':<28}  ${cloud_cost:>6.2f}")
print(f"  {'Local (energy cost only)':<28}  ${local_cost:>6.4f}  ({cloud_cost/local_cost:.0f}\u00d7 cheaper)")
print()

if DRONE_MODE:
    print("  TARGET DRONES  (ScA=state-engine-only  ScB=onboard LLM)")
    print("  " + "─" * 60)
    ram_floor_mb = gguf_mb + 100
    for cls, drone, wg, ram_mb, tks in [
        ("Micro <250g",  "DJI Mini 4 Pro",      512,   2),
        ("Consumer",     "DJI Mavic 3",         4096,  12),
        ("Inspection",   "DJI Matrice 30T",     8192,  40),
        ("Enterprise",   "DJI Matrice 350",    16384,  80),
        ("Delivery",     "Zipline P2",         32768, 150),
    ]:
        sc_b = "\u2713" if ram_mb >= ram_floor_mb else "\u2717"
        print(f"  {sc_b}  {drone} ({cls})   ~{tks} tok/s  ScB:{sc_b}")
else:
    print("  TARGET DEVICES")
    ram_floor_mb = gguf_mb + 100
    def fits(r): return "\u2713" if ram_floor_mb < r * 0.85 else "~"
    for name, ram, note in [
        ("Raspberry Pi 4 (4 GB)",    4096, f"~{max(1, CPU_TOK_S//2)} tok/s"),
        ("iPhone 15 Pro (8 GB)",     8192, f"~{MOB_TOK_S*2} tok/s  (CoreML)"),
        (MOB_DEVICE,                 6144, f"~{MOB_TOK_S} tok/s  (NNAPI)"),
        ("Jetson Orin Nano (8 GB)",  8192, f"~{MOB_TOK_S*3} tok/s  (CUDA)"),
        ("Laptop CPU (no GPU)",     32768, f"~{CPU_TOK_S} tok/s"),
    ]:
        print(f"  {fits(ram)}  {name:<30}  {note}")
    print()
    print("  Run with DRONE_MODE=True in Cell 2 to see drone targets instead.")

print()
fp = pack_stats.get("fingerprint", "N/A")
if fp != "N/A":
    print(f"  Container fingerprint : {fp[:32]}...")
print(f"  Output directory      : {OUTPUT_DIR}")
if AXM_PATH.exists():  print(f"  .axm  \u2192 {AXM_PATH}")
if GGUF_PATH.exists(): print(f"  .gguf \u2192 {GGUF_PATH}")
print(f"  {'\u2550'*60}")

In [ ]:
# Cell 8 — AXIOM SRD vs Google QAT comparison
# Auto-enabled for gemma4 models; set SHOW_COMPARE=True in Cell 2 for others.

is_gemma4 = MODEL_KEY.startswith("gemma4")

if not (SHOW_COMPARE or is_gemma4):
    print("Cell 8 skipped — set SHOW_COMPARE=True in Cell 2 to see AXIOM vs Google table.")
else:
    print("\u2550" * 70)
    print("  CELL 8  \u2014  AXIOM SRD vs GOOGLE QAT  (Gemma 4 Mobile)")
    print("\u2500" * 70)

    rows_goog = [
        ("Gemma 4 E2B", "gemma4-e2b", 11.4, 2.9,  1.1,  0.84),
        ("Gemma 4 E4B", "gemma4-e4b", 17.9, 4.5,  2.5,  2.2),
    ]

    print(f"  {'Model':<16}  {'BF16':>6}  {'AXIOM':>7}  {'AXIOM':>7}  "
          f"{'G-Q4_0':>7}  {'G-Mobile':>9}  {'G-MobTxt':>9}  Gap")
    print(f"  {'':16}  {'GB':>6}  {'SRD-4':>7}  {'GGUF':>7}  "
          f"{'4-bit':>7}  {'LiteRT':>9}  {'LiteRT':>9}")
    print("  " + "\u2500" * 82)

    for name, key, bf16, g_q4, g_mob, g_mobt in rows_goog:
        axm_gb  = round(bf16 * (4.5  / 16), 2)
        gguf_gb = round(bf16 * (4.07 / 16), 2)
        mobile_bpw = round(g_mob / bf16 * 16, 2)
        print(f"  {name:<16}  {bf16:>5.1f}G  {axm_gb:>6.2f}G  {gguf_gb:>6.2f}G  "
              f"  {g_q4:>5.1f}G  {g_mob:>8.2f}G  {g_mobt:>8.2f}G  ~{mobile_bpw} bpw needed")

    print()
    print("  AXIOM SRD-4 \u2248 Google Q4_0  (both ~4-bit, comparable compression)")
    print("  Google Mobile gap: ~1.56 bpw  \u2192  needs sub-4-bit + QAT + pruning")
    print()

    print("  COMPRESSION RATIO vs BF16")
    print(f"  {'':32}  {'E2B':>6}  {'E4B':>6}")
    print("  " + "\u2500" * 46)
    for label, e2b_val, e4b_val in [
        ("AXIOM SRD-4 (.axm)",      round(11.4/(11.4*4.5/16), 2),  round(17.9/(17.9*4.5/16), 2)),
        ("AXIOM GGUF Q4_K_M",       round(11.4/(11.4*4.07/16), 2), round(17.9/(17.9*4.07/16), 2)),
        ("Google Q4_0",             round(11.4/2.9, 2),             round(17.9/4.5, 2)),
        ("Google Mobile (LiteRT)",  round(11.4/1.1, 2),             round(17.9/2.5, 2)),
        ("Google Mobile Text-only", round(11.4/0.84, 2),            round(17.9/2.2, 2)),
    ]:
        bar_v = "\u2593" * min(20, round(e2b_val))
        print(f"  {label:<32}  {e2b_val:>5.1f}\u00d7  {e4b_val:>5.1f}\u00d7  {bar_v}")
    print()

    print("  CAPABILITY COMPARISON")
    print("  " + "\u2500" * 68)
    for capability, axiom_val, google_val in [
        ("Weight compression ~4-bit",     "\u2713 AXIOM",                          "\u2713 Google"),
        ("Mobile-tier <2 bpw",            "\u2717 (roadmap: SRD sparse-D8)",      "\u2713 Google LiteRT"),
        ("HMAC proof per layer",          "\u2713 AXIOM",                          "\u2717"),
        ("Tamper detection",              "\u2713 AXIOM fingerprint",              "\u2717"),
        ("Chain of custody (.axm)",       "\u2713 AXIOM",                          "\u2717"),
        ("MET KV cache compression",      "\u2713 9.5\u00d7 on input side",        "\u2717"),
        ("Task-selective token load",     "\u2713 only task METs in KV",           "\u2717 full ctx always"),
        ("EventToken agent gating",       "\u2713 skip unused modules",            "\u2717 full forward pass"),
        ("QRF core pre-wake",             "\u2713 spawn in idle gap",             "\u2717 always-on"),
        ("Sleeping core power",           "\u2713 T0=0.2W idle\u2192T2 on demand","\u2717 fixed 0.8W"),
        ("Skeleton VRAM floor",           "\u2713 15% (10 MB always-on)",         "\u2717 full model locked"),
        ("Dyn. param hydration",          "\u2713 purge+reload per MET",          "\u2717 static alloc"),
        ("Avg VRAM mixed workload",       "\u2713 21 MB (3.2\u00d7 lower)",        "\u2717 68 MB fixed"),
        ("Per-chunk verify on hydrate",   "\u2713 HMAC from .axm chain",          "\u2717"),
        ("Open format (llama.cpp)",       "\u2713 GGUF",                          "\u2717 LiteRT-only"),
        ("Framework-agnostic",            "\u2713",                               "\u2717 Android/iOS only"),
        ("QAT quality boost",             "\u2717 (post-training only)",          "\u2713 trained with quant"),
    ]:
        print(f"  {capability:<32}  {axiom_val:<28}  {google_val}")
    print()
    print("  ROADMAP NOTE")
    print("  " + "\u2500" * 60)
    print("  Mobile gap (~1.56 bpw) closeable via SRD sparse-D8 residual.")
    print("  axm_cli.py --srd-top-k-pct 0.25 targets ~7 bpw today;")
    print("  full sparse packing (E3 compressed) pushes toward 2\u20133 bpw.")
    print("  Governance story (signed proofs + MET) has no Google equivalent.")

In [ ]:
# Cell 9 — Download GGUF to phone
# Two paths:
#   Path A — Colab browser download (then transfer via AirDrop / Drive / USB)
#   Path B — Push to HuggingFace Hub (phone app pulls over WiFi — no cable needed)
#
# Set HF_TOKEN + HF_REPO_ID below to enable Path B.

import os, subprocess, sys
from pathlib import Path

# ── Config (edit if using Path B) ─────────────────────────────────────────
HF_TOKEN    = os.environ.get("HF_TOKEN", "")  # paste hf_... or add as Colab secret
HF_REPO_ID  = ""     # e.g. "yourname/smollm2-135m-srd4-gguf"  (leave empty to skip)
REPO_PRIVATE = False  # True = private HF repo

# ─────────────────────────────────────────────────────────────────────────
print("\u2550" * 70)
print("  CELL 9  \u2014  DOWNLOAD GGUF TO PHONE")
print("\u2500" * 70)

_gguf = GGUF_PATH if "GGUF_PATH" in dir() and GGUF_PATH.exists() else None
if _gguf is None:
    print("  \u2717 No GGUF found \u2014 run Cell 5 first (requires LLAMACPP set in Cell 2)")
else:
    _mb = _gguf.stat().st_size / 1024**2
    print(f"  GGUF : {_gguf.name}  ({_mb:.0f} MB)")
    print()

    # ── PATH A: Colab browser download ─────────────────────────────────────
    print("  PATH A \u2014 Colab browser download")
    print("  " + "\u2500" * 52)
    try:
        from google.colab import files
        print(f"  Triggering download of {_gguf.name} ({_mb:.0f} MB)...")
        files.download(str(_gguf))
        print("  \u2713 Download started in your browser.")
        print()
        print("  Transfer to phone via:")
        print("    \u2022 AirDrop (iPhone) \u2014 share from Files app")
        print("    \u2022 Google Drive / iCloud \u2014 upload then open on phone")
        print("    \u2022 USB cable \u2192 copy to phone storage")
    except ImportError:
        print("  Not on Colab \u2014 GGUF is at:")
        print(f"    {_gguf}")
    except Exception as _e:
        print(f"  Download error: {_e}")
    print()

    # ── PATH B: HuggingFace Hub push ───────────────────────────────────────
    print("  PATH B \u2014 HuggingFace Hub push (phone pulls directly over WiFi)")
    print("  " + "\u2500" * 52)
    if not HF_TOKEN:
        print("  Skipped \u2014 HF_TOKEN not set.")
        print("  Get a free token: https://huggingface.co/settings/tokens")
        print("  Then set HF_TOKEN and HF_REPO_ID above and re-run this cell.")
    elif not HF_REPO_ID:
        print("  Skipped \u2014 HF_REPO_ID not set.")
        print("  Set HF_REPO_ID = 'yourname/model-name' above.")
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "huggingface_hub"], check=True)
        from huggingface_hub import HfApi, create_repo
        _api = HfApi(token=HF_TOKEN)
        print(f"  Creating/verifying repo: {HF_REPO_ID}")
        create_repo(HF_REPO_ID, token=HF_TOKEN, exist_ok=True, private=REPO_PRIVATE)
        print(f"  Uploading {_gguf.name} ({_mb:.0f} MB)...")
        _api.upload_file(
            path_or_fileobj=str(_gguf),
            path_in_repo=_gguf.name,
            repo_id=HF_REPO_ID,
            token=HF_TOKEN,
        )
        if "STATS_JSON" in dir() and STATS_JSON.exists():
            _api.upload_file(
                path_or_fileobj=str(STATS_JSON),
                path_in_repo=STATS_JSON.name,
                repo_id=HF_REPO_ID,
                token=HF_TOKEN,
            )
        _hf_url = f"https://huggingface.co/{HF_REPO_ID}/resolve/main/{_gguf.name}"
        print(f"  \u2713 Uploaded!")
        print(f"  Direct URL: {_hf_url}")
        print()
        print("  On phone \u2014 paste this URL into PocketPal AI or LLM Farm:")
        print(f"    {_hf_url}")
    print()

    # ── WiFi local server ───────────────────────────────────────────────────
    print("  BONUS \u2014 WiFi local server (same network, no cloud, no USB)")
    print("  " + "\u2500" * 52)
    print("  On your computer (same WiFi as phone):")
    print(f"    cd {_gguf.parent}")
    print( "    python3 -m http.server 8080")
    print(f"  Then on phone open:  http://<computer-ip>:8080/{_gguf.name}")
    print("  Find your IP: ifconfig (mac/linux) or ipconfig (windows)")
    print()

# ── Phone app guide ────────────────────────────────────────────────────────
print("  PHONE APPS THAT LOAD GGUF NATIVELY")
print("  " + "\u2500" * 62)
print(f"  {'App':<20}  {'Platform':<16}  Load method")
print("  " + "\u2500" * 62)
for _app, _plat, _how in [
    ("PocketPal AI",    "iOS + Android",  "Local file or HF URL \u2014 free \u2b50 recommended"),
    ("LLM Farm",        "iOS only",       "Local file or HF URL \u2014 free"),
    ("Termux+llama.cpp","Android",        "CLI \u2014 free, needs one-time build step"),
    ("MLC Chat",        "iOS + Android",  "MLC format only (not GGUF \u2014 skip)"),
]:
    print(f"  {_app:<20}  {_plat:<16}  {_how}")
print()
print("  PocketPal AI (recommended):")
print("    Android: play.google.com/store/apps/details?id=com.pocketpalai")
print("    iOS:     apps.apple.com/app/pocketpal-ai/id6502579189")
print()
_mb_str = f"{_gguf.stat().st_size/1024**2:.0f}" if "_gguf" in dir() and _gguf else str(GGUF_MB_EST)
print(f"  SmolLM2-135M Q4_K_M:  ~{_mb_str} MB  |  ~55 tok/s Pixel 7  |  ~89 MB RAM")
print(f"  Fits all modern phones (2 GB+ RAM) with room to spare.")